In [2]:
import sys
sys.path.insert(0, 'src')

from fake_base_station.ng import (
    NGSetupRequestConfig,
    encode_ngsetup_request,
    decode_ngsetup_request,
    print_decoded_structure,
    PCAPTrafficReplayer
)

In [3]:
# Create configuration with default values
config = NGSetupRequestConfig()


In [12]:
# Encode the NGSetupRequest message
encoded = encode_ngsetup_request(config)

print("Encoded hex:")
print(encoded.hex())

# Decode back for verification
pdu2 = decode_ngsetup_request(encoded)
print("\nDecoded structure:")
print_decoded_structure(pdu2)


Encoded hex:
00150032000004001b00080000f1100000066c0052400903006f6375637030310066000d00000000070000f110000000080015400160

Decoded structure:
<NGAP-PDU (CHOICE)>


In [ ]:
from pycrate_mobile.NAS import *

Msg, err = parse_NAS_MO(unhexlify(encoded.hex()))

show(Msg)


## Send packets from file

In [4]:
# Load in pcap file
traffic_replayer = PCAPTrafficReplayer(pcap_file="./test/gnb_ngap.pcap")
traffic_replayer.summary()

'PCAP Traffic Summary:\n  File: test\\gnb_ngap.pcap\n  Total packets: 154\n  NGAP messages: 154\n  Message sizes: [58, 58, 147, 32, 72, 70, 74, 70, 75, 49, 128, 153, 23, 1174, 57, 89, 29, 25, 42, 102, 1271, 23, 124, 210, 44, 36, 25, 49, 106, 1346, 44, 36, 25, 49, 106, 1346, 44, 74, 70, 67, 70, 68, 47, 117, 153, 23, 1098, 57, 89, 29, 25, 42, 102, 1195, 23, 126, 208, 44, 163, 193, 44, 36, 25, 49, 106, 1346, 44, 134, 39, 134, 39, 38, 25, 51, 33, 33, 33, 74, 70, 68, 47, 122, 153, 23, 1098, 57, 89, 163, 207, 44, 126, 194, 44, 135, 39, 135, 39, 135, 39, 135, 39, 135, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 140, 39, 140, 39, 140, 39, 140, 39, 140, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 134, 39, 38, 25, 51, 33, 157, 1358, 60, 57, 89, 31, 24]'

In [ ]:
from fake_base_station.ng import create_sctp_socket, NGAP_SCTP_PORT

HOST = "127.0.0.1"
PORT = NGAP_SCTP_PORT  # 38412 - standard NGAP/SCTP port per 3GPP TS 38.412

# Option 1: blocking replay over SCTP (opens/closes its own socket)
# traffic_replayer.replay_to_sctp(HOST, PORT, speed_factor=1.0)

# Option 2: background thread replay over SCTP
thread = traffic_replayer.replay_sctp_threaded(HOST, PORT, speed_factor=1.0, packet_type="ngap_only")

print(f"SCTP replay started in background (thread alive: {thread.is_alive()})")

# To stop early:
# thread.stop_event.set()

# To wait for completion:
# thread.join()

# Option 3: bring your own SCTP socket (e.g. to reuse an existing connection)
# sock = create_sctp_socket(HOST, PORT)
# thread = traffic_replayer.replay_threaded(sock, speed_factor=1.0)
# thread.join()
# sock.close()


Exception in thread 

SCTP replay started in background (thread alive: True)

Thread-3 (replay_worker)

:


Traceback (most recent call last):
  File "c:\Users\Thomas\Documents\Fake-Base-Station\src\fake_base_station\ng.py", line 204, in create_sctp_socket
    sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM, IPPROTO_SCTP)
  File "c:\Users\Thomas\miniconda3\envs\Fake-Base-Station\Lib\socket.py", line 236, in __init__
    _socket.socket.__init__(self, family, type, proto, fileno)
    ~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: [WinError 10043] The requested protocol has not been configured into the system, or no implementation for it exists

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "c:\Users\Thomas\miniconda3\envs\Fake-Base-Station\Lib\threading.py", line 1081, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "c:\Users\Thomas\miniconda3\envs\Fake-Base-Station\Lib\threading.py", line 1023, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~

: 